# Surface Code

Working on making a nice interface for the d=3 surface codes from Tomita & Svore 2014.

In [1]:
import sys
sys.path.append('..')

In [2]:
from loqs.codepacks import d3_surface_code as codepack

## Stabilizer Plaquette Template

One of the nice things about the surface code is that the stabilizers can be described in a tileable way.

<img src="images/TomitaSvoreFig2.png" alt="Tomita & Svore 2014 Fig 2" width="400"/>
<img src="images/TomitaSvoreFig4b.png" alt="Tomita & Svore 2014 Fig 4b" width="400"/>

(Reproduced from Tomita & Svore 2014)

We can take advantage of this by coding up a "template" circuit that we will use to build up the syndrome circuit.
Below we show the `StabilizerPlaquetteFactory` object provided in the surface code codepack, which holds these template circuits and is responsible for substituting the correct qubit values into this template.

In [21]:
# X stabilizer, matching Fig 2a
print(str(codepack.surface_factory.template_circuit_dict['X']))

Qubit aux ---|Gh|-| Cb |-| Ca |-| Cd |-| |-| Cc |-|Gh|-|Iz|---
Qubit a   ---|  |-|    |-|Taux|-|    |-| |-|    |-|  |-|  |---
Qubit b   ---|  |-|Taux|-|    |-|    |-| |-|    |-|  |-|  |---
Qubit c   ---|  |-|    |-|    |-|    |-| |-|Taux|-|  |-|  |---
Qubit d   ---|  |-|    |-|    |-|Taux|-| |-|    |-|  |-|  |---



In [22]:
# Z stabilizer, matching Fig 2b
print(str(codepack.surface_factory.template_circuit_dict['Z']))

Qubit aux ---| |-| Tb |-| Ta |-| Td |-| |-| Tc |-| |-|Iz|---
Qubit a   ---| |-|    |-|Caux|-|    |-| |-|    |-| |-|  |---
Qubit b   ---| |-|Caux|-|    |-|    |-| |-|    |-| |-|  |---
Qubit c   ---| |-|    |-|    |-|    |-| |-|Caux|-| |-|  |---
Qubit d   ---| |-|    |-|    |-|Caux|-| |-|    |-| |-|  |---



In [24]:
# Alternate Z stabilizer, matching Fig 4b
print(str(codepack.surface_factory.template_circuit_dict['Zalt']))

Qubit aux ---| |-| Tb |-| |-| Td |-| Ta |-| Tc |-| |-|Iz|---
Qubit a   ---| |-|    |-| |-|    |-|Caux|-|    |-| |-|  |---
Qubit b   ---| |-|Caux|-| |-|    |-|    |-|    |-| |-|  |---
Qubit c   ---| |-|    |-| |-|    |-|    |-|Caux|-| |-|  |---
Qubit d   ---| |-|    |-| |-|Caux|-|    |-|    |-| |-|  |---



The `StabilizerPlaquetteFactory` can be used to get the actual circuits by providing the template type and qubit labels. <font color="red"> Note that the qubit labels have to match the template line labels exactly. </font>

In [25]:
c = codepack.surface_factory.get_circuit("X", ["A0", "D0", "D1", "D2", "D3"])
print(str(c))

Qubit A0 ---|Gh|-|CD1|-|CD0|-|CD3|-| |-|CD2|-|Gh|-|Iz|---
Qubit D0 ---|  |-|   |-|TA0|-|   |-| |-|   |-|  |-|  |---
Qubit D1 ---|  |-|TA0|-|   |-|   |-| |-|   |-|  |-|  |---
Qubit D2 ---|  |-|   |-|   |-|   |-| |-|TA0|-|  |-|  |---
Qubit D3 ---|  |-|   |-|   |-|TA0|-| |-|   |-|  |-|  |---



There are a few advanced modes that can be used. We can also drop certain CNOTs in the case of lower weight stabilizer checks that maintain the same schedule to avoid CNOT collisions, or the midcircuit measurement can be removed in case one wants to generate the syndrome preparation circuit rather than the syndrome extraction circuit.

In [26]:
# Pass in Nones to skip some checks
c = codepack.surface_factory.get_circuit("X", ["A0", "D0", None, "D2", None])
print(str(c))

Qubit A0 ---|Gh|-| |-|CD0|-| |-| |-|CD2|-|Gh|-|Iz|---
Qubit D0 ---|  |-| |-|TA0|-| |-| |-|   |-|  |-|  |---
Qubit D2 ---|  |-| |-|   |-| |-| |-|TA0|-|  |-|  |---



In [27]:
# Can omit the instrument
c = codepack.surface_factory.get_circuit("X", ["A0", "D0", "D1", "D2", "D3"], omit_gates='Iz')
print(str(c))

Qubit A0 ---|Gh|-|CD1|-|CD0|-|CD3|-| |-|CD2|-|Gh|-| |---
Qubit D0 ---|  |-|   |-|TA0|-|   |-| |-|   |-|  |-| |---
Qubit D1 ---|  |-|TA0|-|   |-|   |-| |-|   |-|  |-| |---
Qubit D2 ---|  |-|   |-|   |-|   |-| |-|TA0|-|  |-| |---
Qubit D3 ---|  |-|   |-|   |-|TA0|-| |-|   |-|  |-| |---



## Syndrome Circuit Generation

A `SyndromeCircuit` can now be quickly specified by giving the factory (from above) and the stabilizer types and qubit lists, which is essentially the information provided in the figure below.

<img src="images/TomitaSvoreFig1.png" alt="Tomita & Svore 2014 Fig 1" width="400"/>

(Reproduced from Tomita & Svore 2014)

Below we show examples of the stabilizer specifications given in Surface-17 and Surface-13.
In Surface-17, all stabilizers can be done at once because each stabilizer has a dedicated auxiliary qubit. In Surface-13, there are two phases: the weight-4 checks can be done in parallel, as can the weight-2, but they reuse the same auxiliary qubits. That is specified as well.

In [30]:
# All checks in one stage for Surface-17
syndrome_circ_surface17 = codepack.surface17_syndrome
syndrome_circ_surface17.stabilizer_stages

[{'X': [['A9', None, None, 'D1', 'D2'],
   ['A11', 'D0', 'D1', 'D3', 'D4'],
   ['A14', 'D4', 'D5', 'D7', 'D8'],
   ['A16', 'D6', 'D7', None, None]],
  'Z': [['A10', None, 'D0', None, 'D3'],
   ['A12', 'D1', 'D2', 'D4', 'D5'],
   ['A13', 'D3', 'D4', 'D6', 'D7'],
   ['A15', 'D5', None, 'D8', None]]}]

In [31]:
# Two stages for Surface-13
syndrome_circ_surface13 = codepack.surface13_syndrome
syndrome_circ_surface13.stabilizer_stages

[{'X': [['A9', 'D0', 'D1', 'D3', 'D4'], ['A12', 'D4', 'D5', 'D7', 'D8']],
  'Z': [['A10', 'D1', 'D2', 'D4', 'D5'], ['A11', 'D3', 'D4', 'D6', 'D7']]},
 {'X': [['A10', 'D1', 'D2', None, None], ['A11', None, None, 'D6', 'D7']],
  'Z': [['A9', 'D0', None, 'D3', None], ['A12', None, 'D5', None, 'D8']]}]

Finally, we can generate the full syndrome extraction circuits.

In [11]:
# Optional code to generate Qiskit circuit drawings
#def draw(circuit):
#    import qiskit.qasm2
#    qasm = circuit.convert_to_openqasm(qubit_conversion={q:i for i,q in enumerate(circuit.line_labels)})
#    qiskit_circuit = qiskit.qasm2.loads(qasm)
#    display(qiskit_circuit.draw(output='mpl', style='clifford'))

In [33]:
# Single stage for Surface-17
c17 = syndrome_circ_surface17.get_circuit(codepack.surface17_qubits)

In [34]:
print(str(c17))
#display(draw(c17))

Qubit D0  ---|  |-|CA10|-|TA11|-|    |-|    |-|  |-|  |---
Qubit D1  ---|  |-|TA11|-|CA12|-|    |-|TA9 |-|  |-|  |---
Qubit D2  ---|  |-|CA12|-|    |-|TA9 |-|    |-|  |-|  |---
Qubit D3  ---|  |-|    |-|CA13|-|CA10|-|TA11|-|  |-|  |---
Qubit D4  ---|  |-|CA13|-|TA14|-|TA11|-|CA12|-|  |-|  |---
Qubit D5  ---|  |-|TA14|-|CA15|-|CA12|-|    |-|  |-|  |---
Qubit D6  ---|  |-|    |-|TA16|-|    |-|CA13|-|  |-|  |---
Qubit D7  ---|  |-|TA16|-|    |-|CA13|-|TA14|-|  |-|  |---
Qubit D8  ---|  |-|    |-|    |-|TA14|-|CA15|-|  |-|  |---
Qubit A9  ---|Gh|-|    |-|    |-|CD2 |-|CD1 |-|Gh|-|Iz|---
Qubit A10 ---|  |-|TD0 |-|    |-|TD3 |-|    |-|  |-|Iz|---
Qubit A11 ---|Gh|-|CD1 |-|CD0 |-|CD4 |-|CD3 |-|Gh|-|Iz|---
Qubit A12 ---|  |-|TD2 |-|TD1 |-|TD5 |-|TD4 |-|  |-|Iz|---
Qubit A13 ---|  |-|TD4 |-|TD3 |-|TD7 |-|TD6 |-|  |-|Iz|---
Qubit A14 ---|Gh|-|CD5 |-|CD4 |-|CD8 |-|CD7 |-|Gh|-|Iz|---
Qubit A15 ---|  |-|    |-|TD5 |-|    |-|TD8 |-|  |-|Iz|---
Qubit A16 ---|Gh|-|CD7 |-|CD6 |-|    |-|    |-|Gh|-|Iz|-

In [35]:
c13 = syndrome_circ_surface13.get_circuit(codepack.surface13_qubits)

In [36]:
# We can see the two phases of Surface-13
print(str(c13))
#display(draw(c13))

Qubit D0  ---|  |-|    |-|TA9 |-|    |-|    |-|  |-|  |-|  |-|    |-|CA9 |-|    |-|    |-|  |-|  |---
Qubit D1  ---|  |-|TA9 |-|CA10|-|    |-|    |-|  |-|  |-|  |-|    |-|TA10|-|    |-|    |-|  |-|  |---
Qubit D2  ---|  |-|CA10|-|    |-|    |-|    |-|  |-|  |-|  |-|TA10|-|    |-|    |-|    |-|  |-|  |---
Qubit D3  ---|  |-|    |-|CA11|-|    |-|TA9 |-|  |-|  |-|  |-|    |-|    |-|    |-|CA9 |-|  |-|  |---
Qubit D4  ---|  |-|CA11|-|TA12|-|TA9 |-|CA10|-|  |-|  |-|  |-|    |-|    |-|    |-|    |-|  |-|  |---
Qubit D5  ---|  |-|TA12|-|    |-|CA10|-|    |-|  |-|  |-|  |-|CA12|-|    |-|    |-|    |-|  |-|  |---
Qubit D6  ---|  |-|    |-|    |-|    |-|CA11|-|  |-|  |-|  |-|    |-|    |-|    |-|TA11|-|  |-|  |---
Qubit D7  ---|  |-|    |-|    |-|CA11|-|TA12|-|  |-|  |-|  |-|    |-|    |-|TA11|-|    |-|  |-|  |---
Qubit D8  ---|  |-|    |-|    |-|TA12|-|    |-|  |-|  |-|  |-|    |-|    |-|CA12|-|    |-|  |-|  |---
Qubit A9  ---|Gh|-|CD1 |-|CD0 |-|CD4 |-|CD3 |-|Gh|-|Iz|-|  |-|    |-|TD0 |-|    |-

We can also do some advanced things. We can ask pyGSTi to parallelize/compress the circuit if we wanted (we likely want to do this manually, but hey the capability is there), or we can remove the midcircuit measurements to get the syndrome preparation circuit (could be useful for Harper&Flammia 2023-style benchmarking where 2 syndrome preps = synthetic idle).

In [38]:
c = syndrome_circ_surface13.get_circuit(codepack.surface13_qubits, compress_within_stages=True)
print(str(c))

Qubit D0  ---|    |-|    |-|TA9 |-|    |-|    |-|  |-|  |-|CA9 |-|    |-|    |-|  |-|  |---
Qubit D1  ---|    |-|TA9 |-|CA10|-|    |-|    |-|  |-|  |-|    |-|    |-|TA10|-|  |-|  |---
Qubit D2  ---|CA10|-|    |-|    |-|    |-|    |-|  |-|  |-|    |-|TA10|-|    |-|  |-|  |---
Qubit D3  ---|    |-|CA11|-|    |-|    |-|TA9 |-|  |-|  |-|    |-|CA9 |-|    |-|  |-|  |---
Qubit D4  ---|CA11|-|    |-|TA12|-|TA9 |-|CA10|-|  |-|  |-|    |-|    |-|    |-|  |-|  |---
Qubit D5  ---|    |-|TA12|-|    |-|CA10|-|    |-|  |-|  |-|CA12|-|    |-|    |-|  |-|  |---
Qubit D6  ---|    |-|    |-|    |-|CA11|-|    |-|  |-|  |-|    |-|    |-|TA11|-|  |-|  |---
Qubit D7  ---|    |-|    |-|CA11|-|    |-|TA12|-|  |-|  |-|    |-|TA11|-|    |-|  |-|  |---
Qubit D8  ---|    |-|    |-|    |-|TA12|-|    |-|  |-|  |-|    |-|CA12|-|    |-|  |-|  |---
Qubit A9  ---| Gh |-|CD1 |-|CD0 |-|CD4 |-|CD3 |-|Gh|-|Iz|-|TD0 |-|TD3 |-| Iz |-|  |-|  |---
Qubit A10 ---|TD2 |-|    |-|TD1 |-|TD5 |-|TD4 |-|Iz|-|  |-| Gh |-|CD2 |-|CD1 |-|

In [39]:
c = syndrome_circ_surface13.get_circuit(codepack.surface13_qubits, compress_within_stages=True,
                                        compress_between_stages=True)
print(str(c))

Qubit D0  ---|    |-|    |-|TA9 |-|    |-|    |-|  |-|    |-|CA9 |-|    |-|  |-|  |---
Qubit D1  ---|    |-|TA9 |-|CA10|-|    |-|    |-|  |-|    |-|    |-|TA10|-|  |-|  |---
Qubit D2  ---|CA10|-|    |-|    |-|    |-|    |-|  |-|    |-|TA10|-|    |-|  |-|  |---
Qubit D3  ---|    |-|CA11|-|    |-|    |-|TA9 |-|  |-|    |-|    |-|CA9 |-|  |-|  |---
Qubit D4  ---|CA11|-|    |-|TA12|-|TA9 |-|CA10|-|  |-|    |-|    |-|    |-|  |-|  |---
Qubit D5  ---|    |-|TA12|-|    |-|CA10|-|    |-|  |-|    |-|CA12|-|    |-|  |-|  |---
Qubit D6  ---|    |-|    |-|    |-|CA11|-|    |-|  |-|    |-|TA11|-|    |-|  |-|  |---
Qubit D7  ---|    |-|    |-|CA11|-|    |-|TA12|-|  |-|TA11|-|    |-|    |-|  |-|  |---
Qubit D8  ---|    |-|    |-|    |-|TA12|-|    |-|  |-|    |-|    |-|CA12|-|  |-|  |---
Qubit A9  ---| Gh |-|CD1 |-|CD0 |-|CD4 |-|CD3 |-|Gh|-| Iz |-|TD0 |-|TD3 |-|Iz|-|  |---
Qubit A10 ---|TD2 |-|    |-|TD1 |-|TD5 |-|TD4 |-|Iz|-| Gh |-|CD2 |-|CD1 |-|Gh|-|Iz|---
Qubit A11 ---|TD4 |-|TD3 |-|TD7 |-|TD6 |-| 

In [40]:
c = syndrome_circ_surface17.get_circuit(codepack.surface17_qubits, omit_gates='Iz')
print(str(c))

Qubit D0  ---|  |-|CA10|-|TA11|-|    |-|    |-|  |---
Qubit D1  ---|  |-|TA11|-|CA12|-|    |-|TA9 |-|  |---
Qubit D2  ---|  |-|CA12|-|    |-|TA9 |-|    |-|  |---
Qubit D3  ---|  |-|    |-|CA13|-|CA10|-|TA11|-|  |---
Qubit D4  ---|  |-|CA13|-|TA14|-|TA11|-|CA12|-|  |---
Qubit D5  ---|  |-|TA14|-|CA15|-|CA12|-|    |-|  |---
Qubit D6  ---|  |-|    |-|TA16|-|    |-|CA13|-|  |---
Qubit D7  ---|  |-|TA16|-|    |-|CA13|-|TA14|-|  |---
Qubit D8  ---|  |-|    |-|    |-|TA14|-|CA15|-|  |---
Qubit A9  ---|Gh|-|    |-|    |-|CD2 |-|CD1 |-|Gh|---
Qubit A10 ---|  |-|TD0 |-|    |-|TD3 |-|    |-|  |---
Qubit A11 ---|Gh|-|CD1 |-|CD0 |-|CD4 |-|CD3 |-|Gh|---
Qubit A12 ---|  |-|TD2 |-|TD1 |-|TD5 |-|TD4 |-|  |---
Qubit A13 ---|  |-|TD4 |-|TD3 |-|TD7 |-|TD6 |-|  |---
Qubit A14 ---|Gh|-|CD5 |-|CD4 |-|CD8 |-|CD7 |-|Gh|---
Qubit A15 ---|  |-|    |-|TD5 |-|    |-|TD8 |-|  |---
Qubit A16 ---|Gh|-|CD7 |-|CD6 |-|    |-|    |-|Gh|---

